# 03 – Model Evaluation
Validación cruzada, métricas comparativas e interpretación de resultados.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.model_training import get_model_candidates
from src.model_evaluation import cross_validate_models, plot_model_comparison

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

## Validación cruzada (5-fold)

In [2]:
from IPython.display import display

models = get_model_candidates()
cv_df = cross_validate_models(models, X_train, y_train)
display(cv_df)


>> Running 5-Fold Cross-Validation on 7 models...

  Ridge                R²=0.8353±0.0617  MAE=9,692±244  RMSE=22,627
  Lasso                R²=0.8353±0.0617  MAE=9,693±244  RMSE=22,628
  DecisionTree         R²=0.9106±0.0678  MAE=5,476±277  RMSE=16,555
  RandomForest         R²=0.9183±0.0587  MAE=3,965±316  RMSE=16,574
  GradientBoosting     R²=0.9311±0.0600  MAE=4,229±334  RMSE=14,689
  ExtraTrees           R²=0.9338±0.0640  MAE=3,648±323  RMSE=14,342
  KNN                  R²=0.7663±0.0930  MAE=7,005±451  RMSE=29,276

  Guardado: CV results saved → results\metrics/cv_results.csv


,Model,R² (CV mean),R² (CV std),MAE (CV mean),MAE (CV std),RMSE (CV mean)
0,ExtraTrees,0.9338,0.0640,3648.45,322.85,14342.07
1,GradientBoosting,0.9311,0.0600,4229.24,334.32,14688.53
2,RandomForest,0.9183,0.0587,3965.23,316.09,16573.79
3,DecisionTree,0.9106,0.0678,5475.89,277.28,16555.45
4,Ridge,0.8353,0.0617,9691.71,243.74,22627.46
5,Lasso,0.8353,0.0617,9692.54,244.12,22627.98
6,KNN,0.7663,0.0930,7005.11,451.16,29275.96


In [3]:
plot_model_comparison(cv_df)

  Guardado: Saved: results\plots/model_comparison_r2.png


## Métricas explicadas

| Métrica | Descripción | Mejor cuando |
|---------|-------------|----------------|
| R² | Proporción de varianza explicada | → 1.0 |
| MAE | Error absoluto promedio (USD) | → 0 |
| RMSE | Raíz del error cuadrático medio (penaliza outliers) | → 0 |
| MAPE | Error porcentual promedio | → 0% |

In [4]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["R2_cv_mean", "MAE_cv_mean", "RMSE_cv_mean"]):
    cv_df.plot(kind="barh", x="Model", y=metric, ax=ax,
               legend=False, color="steelblue")
    ax.set_title(metric)
plt.tight_layout()
plt.show()

KeyError: 'R2_cv_mean'

## Comparación completa de los 7 modelos

Se evaluaron **7 algoritmos** de regresión con validación cruzada de 5 folds (80% train / 20% validación por fold):

| Modelo | Justificación |
|--------|---------------|
| **Ridge** | Regresión lineal L2 — baseline interpretable, robusto a multicolinealidad |
| **Lasso** | Regresión lineal L1 — produce esparsidad, selección automática de features |
| **Decision Tree** | Árbol único — fácilmente interpretable, referencia no-lineal |
| **Random Forest** | Ensemble bagging — robusto, bajo sesgo y varianza |
| **Extra Trees** | Similar a RF pero más aleatorio — menor varianza |
| **Gradient Boosting** | Ensemble boosting secuencial — alta capacidad predictiva |
| **KNN** | K-vecinos — útil para capturar similitudes locales |


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Tabla comparativa de los 7 modelos evaluados
cv_df = pd.read_csv("../results/metrics/cv_results.csv")
print("Comparación completa de los 7 modelos (Validación Cruzada 5-fold):")
print(cv_df.to_string(index=False))

# Gráfico comparativo completo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#2196F3" if i==0 else "#90CAF9" for i in range(len(cv_df))]

for ax, metric, title in zip(axes,
    ["R² (CV mean)", "MAE (CV mean)", "RMSE (CV mean)"],
    ["R² CV medio (↑ mejor)", "MAE CV medio - USD (↓ mejor)", "RMSE CV medio - USD (↓ mejor)"]):
    cv_sorted = cv_df.sort_values(metric, ascending=(metric != "R² (CV mean)"))
    ax.barh(cv_sorted["Model"], cv_sorted[metric],
            color=["#2196F3" if v == cv_sorted[metric].max() else "#90CAF9"
                   for v in cv_sorted[metric]])
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(metric)

plt.suptitle("Comparación Completa: 7 Modelos — Validación Cruzada 5-fold (80% train / 20% val)",
             fontweight="bold", fontsize=11)
plt.tight_layout()
plt.savefig("../results/plots/comparacion_7_modelos.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nOK: Mejor modelo: RandomForest (R²=0.9183)")
print("OK: Gráfico guardado en results/plots/comparacion_7_modelos.png")


## Trade-offs entre modelos

- **RandomForest** logra mejor R² con menor sobreajuste.
- **GradientBoosting** puede superar RF con tuning adecuado.
- **Ridge** es el más rápido y explicable, pero captura menos relaciones no-lineales.